# Agentic AI - Email Assistant

### Import libraries and initialise ai suite Client

In [1]:
from dotenv import load_dotenv
import aisuite as ai
import json
import sys

sys.path.append(".emailserver")

import emailserver.display_functions as df
import utils
import emailserver.email_service as es
import emailserver.email_utils as eu

load_dotenv()
client = ai.Client()

Using EMAIL_SERVER_API_URL: http://127.0.0.1:8000


In [2]:
# uncomment the line 'utils.test_*' you want to try
new_email_id = utils.test_send_email()
#print("Sent email ID:", new_email_id)
_ = utils.test_get_email(new_email_id['id'])
_ = utils.test_list_emails()
_ = utils.test_filter_emails(recipient="test@example.com")
_ = utils.test_search_emails("lunch")
_ = utils.test_unread_emails()
_ = utils.test_mark_read(new_email_id['id'])
_ = utils.test_mark_unread(new_email_id['id'])
_ = utils.test_delete_email(new_email_id['id'])
_ = utils.reset_database()

------------------------Preloading sample emails into the database…
http://127.0.0.1:8000
Preloading sample emails into the database…
http://127.0.0.1:8000/send/
Response status code: 200
Response status code: b'{"id":7,"sender":"you@email.com","recipient":"test@example.com","subject":"Test Subject","body":"This is a test email body.","timestamp":"2025-11-19T18:09:02.544863","is_read":false}'


Filter params: {'recipient': 'test@example.com'}


### Define the agent prompt

In [3]:
def build_prompt(request_: str) -> str:
    return f"""
- You are an AI assistant specialized in managing emails.
- You can perform various actions such as listing, searching, filtering, and manipulating emails.
- Use the provided tools to interact with the email system.
- Never ask the user for confirmation before performing an action.
- If needed, my email address is "you@email.com" so you can use it to send emails or perform actions related to my account.

{request_.strip()}
"""

In [4]:
example_prompt = build_prompt("Delete the Happy Hour email")
utils.print_html(content=example_prompt, title="Example example_prompt")

In [8]:
utils.reset_database()

{'message': 'Database reset successfully'}

In [6]:
# Try your own requests
prompt_ = build_prompt("Check for unread emails from boss@email.com, mark them as read, and send a polite follow-up.")

response = client.chat.completions.create(
    model="openai:gpt-4.1", # LLM
    messages=[{"role": "user", "content": (
        prompt_
    )}],
    tools=[ # list of tools that the LLM can access
        eu.search_unread_from_sender,
        eu.list_unread_emails,
        eu.search_emails,
        eu.get_email,
        eu.mark_email_as_read,
        eu.send_email
    ],
    max_turns=5,
)

df.pretty_print_chat_completion(response)

### Missing Tool

In [11]:
prompt = build_prompt("Delete all emails from alice@work.com. do not append from: to the search query")

response = client.chat.completions.create(
    model="openai:gpt-4.1", # LLM
    messages=[{"role": "user", "content": (
        prompt
    )}],
    tools=[ # list of tools that the LLM can access
        eu.search_unread_from_sender,
        eu.list_unread_emails,
        eu.search_emails,
        eu.get_email,
        eu.mark_email_as_read,
        eu.send_email,
        eu.delete_email
    ],
    max_turns=5,
)

df.pretty_print_chat_completion(response)

In [12]:
prompt = build_prompt("Delete all Happy Hour emails")

response = client.chat.completions.create(
    model="openai:gpt-4.1", # LLM
    messages=[{"role": "user", "content": (
        prompt
    )}],
    tools=[ # list of tools that the LLM can access
        eu.search_unread_from_sender,
        eu.list_unread_emails,
        eu.search_emails,
        eu.get_email,
        eu.mark_email_as_read,
        eu.send_email,
        eu.delete_email
    ],
    max_turns=5,
)

df.pretty_print_chat_completion(response)